# Encoding benchmark

Compare encoding bases for a maximally entangled two-qutrit graph state.

This example reuses **41 saved theta gate sets** and recompiles them for the selected backend. Costs include routing. No QPU jobs are submitted.

Install from the repository root with `python -m pip install -e .`, then **Run All**.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/qudits_on_qubits").is_dir())
sys.path[:0] = [str(root / "src"), str(root / "examples")]

from encoding_benchmark_demo import load_theta_demo, plot_pareto
from qudits_on_qubits.benchmarks import (
    BackendTarget, BenchmarkConfig, LocalSU2, SchmidtTheta, OptimizedSynthesis,
    run_benchmark, two_qutrit_graph_circuit,
)

## 1. Choose the backend and candidates

In [ ]:
backend = BackendTarget.local(
    num_qubits=4,
    coupling_map=[(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2)],
)
# backend = BackendTarget.iqm("garnet")  # Or "emerald".
# backend = BackendTarget.ibm("your_backend")

demo = load_theta_demo(root / "examples/data/theta_demo.zip")
families, synthesis = [demo.family], demo.synthesis

# For a new search (slower), replace the line above with:
# families = [LocalSU2(samples=4, seed=42), SchmidtTheta(points=5)]
# synthesis = OptimizedSynthesis()
output_dir = None  # Optional: a new directory for this run.

## 2. Run the benchmark

In [ ]:
result = run_benchmark(
    two_qutrit_graph_circuit(),
    families=families,
    backend=backend,
    synthesis=synthesis,
    config=BenchmarkConfig(transpiler_seeds=(0, 1, 2), cz3_tolerance=5e-4),
    output_dir=output_dir,
)

## 3. Compare with the baseline

Lower is better. Pareto uses mean 2q count, mean depth and depth standard deviation. Results apply to this backend and compilation setup.

[Full documentation](../docs/encoding_benchmark.md)

In [ ]:
columns = ["candidate_name", "mean_two_qubit_gate_count", "mean_depth", "std_depth", "pareto_rank"]
stats = result.statistics
summary = stats.loc[stats.is_baseline | stats.pareto_rank.eq(1), columns]
display(summary.round(3))
print(f"{result.trials.success.sum()}/{len(result.trials)} trials accepted")
print("Saved results:", result.output_dir)

In [ ]:
display(plot_pareto(result))